# Check QAChat Tutorials

This auxiliary notebook executes every notebook in `tutorials/` in numerical
order and reports whether it completes successfully. It is intended to live
in `assets/`.

The original tutorial notebooks are not modified. Executed copies and command
logs are written below `sandbox/tutorial_checks/` so failures can be inspected
afterwards.

Some tutorials load or download language and embedding models. A complete
verification therefore requires the normal QAChat environment and can take
substantially longer than the purely deterministic tutorials.


In [ ]:
from __future__ import annotations


## Locate and prepare the repository

The check uses the same repository environment expected by the tutorials
themselves.


In [ ]:
from pathlib import Path


def find_repository_root(start: Path | None = None) -> Path:
    """Find the QAChat repository root.

    Parameters
    ----------
    start: pathlib.Path | None, optional
        Directory from which to begin the search.

    Returns
    -------
    pathlib.Path
        Absolute path to the QAChat repository root.

    Raises
    ------
    RuntimeError
        If the repository root cannot be found.

    """

    current = (start or Path.cwd()).resolve()
    for directory in (current, *current.parents):
        if (directory / "qachat").is_dir():
            return directory

    raise RuntimeError(
        "Could not locate the QAChat repository root "
        f"when starting from {current}."
    )


In [ ]:
import sys


repository_root = find_repository_root()

if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

from setup.notebook import prepare_notebook

repository_root = prepare_notebook(repository_root)
tutorial_directory = repository_root / "tutorials"
output_directory = repository_root / "sandbox/tutorial_checks"

print(f"Repository root: {repository_root}")
print(f"Tutorial directory: {tutorial_directory}")
print(f"Check output: {output_directory}")


## Collect tutorials

Tutorials are ordered by the numeric prefix in their filename, so `100_...`
follows `90_...` rather than `10_...`.


In [ ]:
import re


def tutorial_sort_key(path: Path) -> tuple[int, str]:
    """Return the numerical ordering key for one tutorial.

    Parameters
    ----------
    path: pathlib.Path
        Tutorial notebook path.

    Returns
    -------
    tuple[int, str]
        Numeric prefix followed by the filename.

    """

    match = re.match(r"(\d+)_", path.name)
    number = int(match.group(1)) if match else sys.maxsize
    return number, path.name


In [ ]:
tutorials = sorted(
    tutorial_directory.glob("*.ipynb"),
    key=tutorial_sort_key,
)

if not tutorials:
    raise RuntimeError("No tutorial notebooks were found.")

print(f"Found {len(tutorials)} tutorials:")
for tutorial in tutorials:
    print(f"- {tutorial.name}")


## Execute all tutorials

Each tutorial is executed in a separate Jupyter process. This avoids state
leaking from one notebook into the next. The tutorial's own setup cell remains
active, so a wrong kernel, missing dependency, missing source asset, or runtime
error causes that tutorial to fail.


In [ ]:
CELL_TIMEOUT_SECONDS = 1800
PROCESS_TIMEOUT_SECONDS = 3600


In [ ]:
import shutil
import subprocess
import time


if output_directory.exists():
    shutil.rmtree(output_directory)
output_directory.mkdir(parents=True)

results = []

for tutorial in tutorials:
    started = time.perf_counter()
    command = [
        sys.executable,
        "-m",
        "jupyter",
        "nbconvert",
        "--to",
        "notebook",
        "--execute",
        f"--ExecutePreprocessor.timeout={CELL_TIMEOUT_SECONDS}",
        "--output-dir",
        str(output_directory),
        "--output",
        tutorial.name,
        str(tutorial),
    ]

    try:
        completed = subprocess.run(
            command,
            cwd=repository_root,
            capture_output=True,
            text=True,
            timeout=PROCESS_TIMEOUT_SECONDS,
            check=False,
        )
        return_code = completed.returncode
        log_text = completed.stdout + completed.stderr
    except subprocess.TimeoutExpired as error:
        return_code = -1
        stdout = error.stdout or ""
        stderr = error.stderr or ""
        log_text = f"{stdout}{stderr}\nProcess timed out."

    elapsed = time.perf_counter() - started
    passed = return_code == 0
    status = "PASS" if passed else "FAIL"
    log_path = output_directory / f"{tutorial.stem}.log"
    log_path.write_text(log_text, encoding="utf-8")

    results.append(
        {
            "tutorial": tutorial.name,
            "passed": passed,
            "return_code": return_code,
            "seconds": elapsed,
            "log": log_path,
        }
    )
    print(f"{status:4}  {elapsed:8.1f} s  {tutorial.name}")


## Verification summary

The final cell raises an error when any tutorial failed, making the notebook
suitable as a simple integration check.


In [ ]:
passed_count = sum(result["passed"] for result in results)
failed = [result for result in results if not result["passed"]]

print("Tutorial verification")
print("---------------------")
print(f"Passed: {passed_count}/{len(results)}")
print(f"Failed: {len(failed)}/{len(results)}")

if failed:
    print("\nFailed tutorials:")
    for result in failed:
        relative_log = result["log"].relative_to(repository_root)
        print(f"- {result['tutorial']} (log: {relative_log})")
    raise RuntimeError(
        f"{len(failed)} tutorial notebook(s) failed."
    )

print("All tutorial notebooks completed successfully.")
